# SO-101 Perception Benchmark — metric-depth models vs known ground truth

Compare single-camera **metric-depth** models (Depth Pro, UniDepthV2) on 48 sim frames whose object centre is known **exactly** in both pixel and world coordinates.

Detection is **out of scope**: SAM already gives the centre pixel perfectly on these frames, so we score depth only, read **at the known GT pixel**. Ground truth is used solely to (a) know which pixel is the centre and (b) score error — it never drives an estimate.

**Run order:** run §0 (setup + logger), install ONE model in §1 and **restart the runtime**, then run §2–§5 top to bottom. The last cell pushes the run log to `logs/`.

## 0. Setup + run logger

In [ ]:
# Clone the repo (first Colab run) and import. camera_math is pure-numpy, no deps.
import os, sys
CLONE_URL = 'https://github.com/Yunsmn/RoboticsPerceptionTest.git'
if not os.path.exists('camera_math.py'):
    os.system('git clone ' + CLONE_URL)
    os.chdir('RoboticsPerceptionTest')
sys.path.insert(0, '.')
import numpy as np, json
from pathlib import Path
from PIL import Image
import camera_math as CM
print('setup OK')

In [ ]:
# Tees every subsequent cell (source + stdout/stderr + tracebacks) to run_log.md.
# This is how the run gets back to the author: run all, then the last cell pushes the log.
import sys, io, datetime, traceback, subprocess
from IPython import get_ipython
_LOG = 'run_log.md'; _f = open(_LOG, 'w')
def _w(s=''):
    _f.write(str(s) + '\n'); _f.flush()
_w('# Run log — ' + datetime.datetime.now().isoformat(timespec='seconds'))
try:
    _gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                          capture_output=True, text=True).stdout.strip()
except Exception:
    _gpu = ''
_w('- GPU: ' + (_gpu or 'none / CPU')); _w('- Python: ' + sys.version.split()[0])
class _Tee:
    _is_tee = True
    def __init__(self, real): self._real = real
    def write(self, s):
        n = self._real.write(s)
        try: _f.write(s); _f.flush()
        except Exception: pass
        return n
    def flush(self):
        try: self._real.flush()
        except Exception: pass
    def __getattr__(self, k):
        return getattr(self.__dict__['_real'], k)
if not getattr(sys.stdout, '_is_tee', False): sys.stdout = _Tee(sys.stdout)
if not getattr(sys.stderr, '_is_tee', False): sys.stderr = _Tee(sys.stderr)
_ip = get_ipython(); _n = {'i': 0}
def _pre(info):
    _n['i'] += 1
    _w(); _w('## Cell ' + str(_n['i'])); _w('```python')
    _w((info.raw_cell or '').rstrip()); _w('```'); _w('**output:**'); _w('```text')
def _post(res):
    _w('```')
    err = getattr(res,'error_in_exec',None) or getattr(res,'error_before_exec',None)
    if err is not None:
        _w('**ERROR:**'); _w('```text')
        _w(''.join(traceback.format_exception(type(err), err, err.__traceback__))); _w('```')
if not globals().get('_LOGGER_ON'):
    _ip.events.register('pre_run_cell', _pre); _ip.events.register('post_run_cell', _post)
    _LOGGER_ON = True
def download_log():
    _f.flush()
    try:
        from google.colab import files; files.download(_LOG)
    except Exception as e:
        print('download failed (%s) - file is at %s' % (e, _LOG))
print('run logger active -> run_log.md   (call download_log() at the end)')

## 1. Install a depth model
Uncomment **one** model, run it, then **Runtime → Restart session** (pip can leave a mismatched numpy). Do **not** re-run this cell after restart.

In [ ]:
# Install ONE metric-depth model (their deps conflict — one per Colab session).
# After it finishes: Runtime -> Restart session, then run every cell below (skip this one).

# --- Depth Pro (Apple) ---
get_ipython().system('pip -q install git+https://github.com/apple/ml-depth-pro.git')
get_ipython().system('mkdir -p checkpoints && wget -q -nc https://ml-site.cdn-apple.com/models/depth-pro/depth_pro.pt -P checkpoints')

# --- UniDepthV2 (ETH) — comment out Depth Pro above, uncomment this instead ---
# get_ipython().system('pip -q install git+https://github.com/lpiccinelli-eth/UniDepth.git')

## 2. Data + harness self-check
`data/manifest.json` carries each frame's `gt_uv` (exact centre pixel), `gt_xyz` (world metres) and `gt_depth_m` (forward-axis depth). The self-check deprojects at the true depth and must recover the true xyz to ~1e-16 — proving the scoring math before any model runs.

In [ ]:
# Load the frames + camera, and PROVE the eval harness with two model-free self-checks.
man = json.loads(Path('data/manifest.json').read_text())
cam = man['camera']
f, cx, cy = cam['f'], cam['cx'], cam['cy']
cam_pos = np.array(cam['cam_pos'], float)
R_cw    = np.array(cam['R_cw'], float)
FRAMES  = man['frames']
print(len(FRAMES), 'frames | %dx%d | f=%.1f' % (cam['width'], cam['height'], f))

def project(P):
    # World point -> (u, v, forward-axis depth m). Inverse of point_at_depth.
    xc, yc, zc = R_cw.T @ (np.asarray(P, float) - cam_pos)
    return cx + f * xc / -zc, cy - f * yc / -zc, -zc

# Check A (the eval math): project gt_xyz then deproject at that depth -> recover gt_xyz.
# Pure inverse, full precision, no model, no manifest rounding -> machine epsilon.
inv = []
for fr in FRAMES:
    P = np.array(fr['gt_xyz'], float); u, v, d = project(P)
    inv.append(np.linalg.norm(CM.point_at_depth(u, v, f, cx, cy, cam_pos, R_cw, d) - P))
print('self-check A  project<->deproject  max err = %.2e m  (want ~1e-16)' % max(inv))

# Check B (the stored data): reprojected centre must match the manifest's stored gt_uv.
px = [np.hypot(*(np.subtract(project(fr['gt_xyz'])[:2], fr['gt_uv']))) for fr in FRAMES]
print('self-check B  stored gt_uv vs reproj  max err = %.4f px' % max(px))

## 3. Depth adapters
Each returns an `(H, W)` metric-depth map in metres along the camera forward axis. Canonical usage of each model's own `infer` API — nothing else.

In [ ]:
# One function per model: RGB image (H,W,3 uint8) -> (H,W) metric depth in metres,
# along the camera forward axis (the convention camera_math.point_at_depth consumes).
import torch
_DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', _DEV)
_M = {}   # load each model once

def depth_pro_map(img_rgb):
    import depth_pro
    if 'dp' not in _M:
        m, tf = depth_pro.create_model_and_transforms(device=torch.device(_DEV))
        _M['dp'] = (m.eval(), tf)
    m, tf = _M['dp']
    with torch.no_grad():
        # f_px MUST be a device tensor: Depth Pro's infer() runs f_px.squeeze() internally,
        # which throws on a Python float. Passing the known focal skips its focal estimation.
        pred = m.infer(tf(img_rgb), f_px=torch.tensor(float(f), device=_DEV))
    return pred['depth'].squeeze().float().cpu().numpy()

def unidepth_map(img_rgb):
    from unidepth.models import UniDepthV2
    if 'ud' not in _M:
        _M['ud'] = UniDepthV2.from_pretrained('lpiccinelli/unidepth-v2-vitl14').to(_DEV).eval()
    rgb = torch.from_numpy(img_rgb).permute(2, 0, 1)                 # C,H,W uint8
    K = torch.tensor([[f, 0, cx], [0, f, cy], [0, 0, 1]], dtype=torch.float32)
    with torch.no_grad():
        pred = _M['ud'].infer(rgb, K)
    return pred['depth'].squeeze().float().cpu().numpy()

## 4. Evaluate
Depth at the known pixel → `point_at_depth` → world xyz → error vs the known centre, over all 48 frames. Also reports the gt/pred depth ratio and the scale-corrected error, to tell a fixable global-scale miss from genuine failure.

In [ ]:
# Score a model at each frame's KNOWN centre pixel: read depth -> deproject -> world error (mm).
# Also diagnose scale: print gt vs predicted depth (+ ratio), and the error AFTER applying one
# global scale (median gt/pred). If raw error is large but scaled error is small, the model's
# depth is right up to a single scale -> one anchor (plane touch / known size) recovers it.
def eval_model(depth_map, name, show=6):
    recs = []
    for fr in FRAMES:
        img = np.array(Image.open('data/' + fr['png']).convert('RGB'))
        u, v = fr['gt_uv']
        d = float(depth_map(img)[int(round(v)), int(round(u))])
        recs.append((u, v, d, np.array(fr['gt_xyz'], float), fr['gt_depth_m'], fr['png']))
    def loc(scale):
        return np.array([np.linalg.norm(CM.point_at_depth(u, v, f, cx, cy, cam_pos, R_cw, d*scale) - g)
                         for (u, v, d, g, gd, pp) in recs]) * 1000
    s = float(np.median([gd / d for (u, v, d, g, gd, pp) in recs if d > 0]))   # global gt/pred scale
    raw, cor = loc(1.0), loc(s)
    print('  %s: gt_depth vs pred_depth (first %d frames)' % (name, show))
    for u, v, d, g, gd, pp in recs[:show]:
        print('    %-16s gt %.3f m   pred %.3f m   ratio %.2f' % (pp, gd, d, (gd / d if d else 0)))
    r = {'n': len(recs), 'loc_median_mm': round(float(np.median(raw)), 1),
         'loc_p95_mm': round(float(np.percentile(raw, 95)), 1), 'median_gt_over_pred': round(s, 3),
         'scaled_loc_median_mm': round(float(np.median(cor)), 1),
         'scaled_loc_p95_mm': round(float(np.percentile(cor, 95)), 1)}
    print('  ->', name, r)
    return r

results = {}
def _run(name, fn):
    try:
        results[name] = eval_model(fn, name)
    except Exception as e:      # keep the other model's number if one backend fails
        results[name] = {'error': '%s: %s' % (type(e).__name__, str(e)[:120])}
        print('%-12s FAILED: %s' % (name, results[name]['error']))

_run('depth_pro', depth_pro_map)        # the model you installed this session
# _run('unidepth_v2', unidepth_map)     # uncomment instead if you installed UniDepth

## 5. Focal probe (Depth Pro)
If the raw error is a clean ~2.4× scale, find the `f_px` that reads natively metric.

In [ ]:
# Focal probe (Depth Pro only): raw gt/pred ratio ~2.4 ~= 1536/640 (Depth Pro's canonical size
# over our image width) suggests f_px is at the wrong resolution. Find the f_px that reads
# natively metric (ratio ~1.0) on the hero frame. Cheap: 3 inferences on one image.
import torch
if 'dp' in _M:
    m, tf = _M['dp']
    fr0 = min(FRAMES, key=lambda fr: abs(fr['gt_xyz'][1]))
    img0 = np.array(Image.open('data/' + fr0['png']).convert('RGB'))
    u0, v0 = fr0['gt_uv']; x0 = tf(img0); gd0 = fr0['gt_depth_m']
    trials = [('f=%.1f native-res' % f, float(f)),
              ('f=None (estimated)', None),
              ('f=%.1f (f*1536/640)' % (f * 1536 / 640), float(f) * 1536 / 640)]
    for tag, fpx in trials:
        kw = {} if fpx is None else {'f_px': torch.tensor(float(fpx), device=_DEV)}
        with torch.no_grad():
            dm = m.infer(x0, **kw)['depth'].squeeze().float().cpu().numpy()
        d0 = float(dm[int(round(v0)), int(round(u0))])
        print('  %-26s pred %.3f m  gt %.3f m  ratio %.2f' % (tag, d0, gd0, (gd0 / d0 if d0 else 0)))
else:
    print('focal probe skipped (Depth Pro not loaded this session)')

## 6. Compare + decide

In [ ]:
# Compare to the pipelines we already measured on this same sim (~2 mm grasp basin).
REFERENCE = {
    'sam_plane (prior)':      1.7,   # FastSAM + known support plane z=0.015 (uses a prior)
    'mono_sam_depth':        12.0,   # FastSAM + Depth-Anything (affine, no plane)
    'triangulation_3cam':     1.1,   # honest multi-view geometry
}
print('--- reference (measured earlier) ---')
for k, v in REFERENCE.items(): print('  %-22s loc_median %5.1f mm' % (k, v))
print('--- this run (single-camera, plane-free metric depth) ---')
for k, v in results.items(): print('  %-22s %s' % (k, v))
print('\nBar: loc_median (ideally p95) <= ~2 mm at the GT pixel => single-shot depth can carry the cube.')

## 7. Push the run log
Copies `run_log.md` → `logs/run_<ts>.md` and pushes it (needs a `GH_TOKEN` Colab secret with repo write). The author pulls it, reads the number, iterates.

In [ ]:
# Push run_log.md -> logs/run_<timestamp>.md on GitHub. Token comes from Colab Secrets.
import subprocess, datetime, os, shutil
from google.colab import userdata
print('=== results snapshot ===')
print('results:', globals().get('results', '(eval cell not run yet)'))
def _sh(args, secret=None):
    r = subprocess.run(args, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if secret and out: out = out.replace(secret, '***')
    if out: print(out)
    return r.returncode
try:
    _TOKEN = userdata.get('GH_TOKEN')
except Exception as _e:
    _TOKEN = None; print('No GH_TOKEN secret:', _e)
if not _TOKEN:
    print('Set GH_TOKEN in the Colab 🔑 Secrets panel, then re-run. (Fallback: download_log())')
else:
    _REPO = 'Yunsmn/RoboticsPerceptionTest'
    try: _f.flush()
    except Exception: pass
    os.makedirs('logs', exist_ok=True)
    _stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    _dest = 'logs/run_%s.md' % _stamp
    shutil.copy('run_log.md', _dest)
    _url = 'https://%s@github.com/%s.git' % (_TOKEN, _REPO)
    _sh(['git','config','user.email','younesosf@gmail.com'])
    _sh(['git','config','user.name','Yunsmn'])
    _sh(['git','pull','--rebase','--autostash', _url, 'main'], secret=_TOKEN)
    _sh(['git','add', _dest])
    _sh(['git','commit','-m','log: run %s' % _stamp])
    _rc = _sh(['git','push', _url, 'HEAD:main'], secret=_TOKEN)
    print(('PUSHED ' if _rc == 0 else 'PUSH FAILED (rc=%d) ' % _rc) + _dest)
    print('-> tell the author: pull and read ' + _dest)